# RMBI3110 — Assignment 2: The Alpha Challenge
## Can You Beat the Market with Machine Learning?

**Professor Xuhu Wan | HKUST Business School | Spring 2026**

**Names & Student IDs:** *(fill in)*

---

**Dataset:** `alpha_dataset_v2.csv` (provided via Google Drive)

| Category | Examples | Count |
|---|---|---|
| Price / Momentum | `ret_1`, `ret_2_12`, `vol_12m`, `beta`, `ivol` | 12 |
| Accounting / Value | `bm`, `ep`, `gpa`, `roe`, `ag`, `lev` | 15 |
| Analyst | `sue`, `revision`, `dispersion`, `beat` | 6 |
| Technical | `rsi_14`, `macd_hist`, `bb_position`, `roc_3` | 13 |
| Options | `iv_atm_30d`, `iv_skew`, `pc_vol_ratio`, `vrp` | 6 |
| Peer / Industry | `peer_sue`, `leader_ret`, `ind_mom` | 17 |
| Quarterly fundamentals | `sue_q`, `rev_surp`, `earn_growth_yoy` | 19 |
| Interactions | `mom_x_size`, `val_x_prof`, `mom_x_vol` | 10 |
| Sector / Macro | `sector_ret_avg`, `macro_unc_1m` | 13 |
| **Total raw features** | | **116** |

**Key columns:**

- `y_xs`: standardized forward monthly return  (training target)
- `y_raw`: raw forward monthly return (portfolio evaluation only)


---
## Setup

In [3]:
!pip install gdown

  Using cached gdown-6.0.0-py3-none-any.whl.metadata (7.4 kB)
Using cached gdown-6.0.0-py3-none-any.whl (18 kB)


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from numpy.linalg import lstsq
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LassoCV
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({'figure.figsize': (13, 5.5), 'font.size': 11})

# -- Load data from Google Drive --
import gdown, os
DATA_FILE = "alpha_dataset_v2.csv"
if not os.path.exists(DATA_FILE):
    print("Downloading dataset from Google Drive...")
    gdown.download(id="1GQG6PxI8alJJABCThLEZk7TFpjjLkY0d", output=DATA_FILE, quiet=False)
df = pd.read_csv(DATA_FILE)
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} cols, {df['ym'].nunique()} months")

# Features: cross-sectionally standardized (_xs)
X_COLS = [c for c in df.columns if c.endswith('_xs') and c != 'y_xs']
print(f"Features: {len(X_COLS)}")

TRAIN_TARGET = 'y_xs' # standardize forward monthly return
EVAL_TARGET = 'y_raw' # raw forward monthly return.
OOS_START = '2005-01'
RETRAIN_EVERY = 12
K = 30

df_sp = df.copy()
print(f"Universe: {df_sp.shape[0]:,} rows, {df_sp['permno'].nunique()} stocks")

# SPY benchmark
ff = df_sp.groupby('ym')[['Mkt_RF', 'rf_ff']].first().sort_index()
ff['spy_ret'] = ff['Mkt_RF'] + ff['rf_ff']
spy_oos = ff.loc[ff.index >= OOS_START, 'spy_ret']
spy_cum = (1 + spy_oos).cumprod()
spy_sr = spy_oos.mean() / spy_oos.std() * np.sqrt(12)
print(f"SPY OOS: {len(spy_oos)} months, SR = {spy_sr:.2f}")

Downloading...
From (original): https://drive.google.com/uc?id=1GQG6PxI8alJJABCThLEZk7TFpjjLkY0d
From (redirected): https://drive.google.com/uc?id=1GQG6PxI8alJJABCThLEZk7TFpjjLkY0d&confirm=t&uuid=82abb1e6-fdcd-4212-ab57-67a9fcce4ea9
To: c:\Users\marti\OneDrive\Desktop\Coding projects\Alpha Model\Code\alpha_dataset_v2.csv
 92%|█████████▏| 856M/936M [03:14<00:21, 3.62MB/s] 

In [ ]:
# -- Helper functions --

def topk_ls(sub, pred_col, K=30):
    """EW top-K long minus bottom-K short."""
    sub = sub.dropna(subset=[pred_col, EVAL_TARGET])
    if len(sub) < 2 * K:
        return np.nan
    top = sub.nlargest(K, pred_col)
    bot = sub.nsmallest(K, pred_col)
    return top[EVAL_TARGET].mean() - bot[EVAL_TARGET].mean()


def walk_forward(df_in, pred_col, K=30):
    """Compute monthly L/S returns OOS."""
    oos = df_in[df_in['ym'] >= OOS_START]
    rets = {}
    for m, grp in oos.groupby('ym'):
        r = topk_ls(grp, pred_col, K)
        if not np.isnan(r):
            rets[m] = r
    return pd.Series(rets).sort_index()


def perf(s, name=''):
    """Compute performance statistics."""
    mu = s.mean() * 12
    vol = s.std() * np.sqrt(12)
    sr = mu / vol if vol > 0 else 0
    cum = (1 + s).cumprod()
    mdd = (cum / cum.cummax() - 1).min()
    return {'Strategy': name, 'SR': round(sr, 2), 'Ann Mean': f'{mu:.1%}',
            'Ann Vol': f'{vol:.1%}', 'MDD': f'{mdd:.1%}',
            'Total': f'{cum.iloc[-1]-1:.0%}'}


def plot_strats(strat_dict, title):
    """Plot cumulative wealth for multiple strategies + SPY."""
    fig, ax = plt.subplots(figsize=(13, 5.5))
    for name, s in strat_dict.items():
        cum = (1 + s).cumprod()
        sr = s.mean() / s.std() * np.sqrt(12) if s.std() > 0 else 0
        ax.plot(cum.index, cum.values, label=f'{name} (SR={sr:.2f})', linewidth=1.3)
    ax.plot(spy_cum.index, spy_cum.values, color='gray', linestyle='--',
            linewidth=2.2, label=f'SPY (SR={spy_sr:.2f})', alpha=0.8)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel('Cumulative Wealth ($1)')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(alpha=0.3)
    # Thin x-axis ticks
    ticks = list(range(0, len(ax.get_xticks()), max(1, len(ax.get_xticks()) // 15)))
    ax.set_xticks([ax.get_xticks()[i] for i in ticks if i < len(ax.get_xticks())])
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()


print("Helper functions loaded.")

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────

def build_features_linear(df_slice):
    """Tier 1: Conservative feature set for Lasso (~25-35 features).
    Core factors + pre-built interactions + hand-crafted composites.
    """
    feat = pd.DataFrame(index=df_slice.index)

    # Core factor features
    core = [
        # Momentum
        'ret_1_xs', 'ret_2_12_xs', 'ret_2_6_xs',
        # Value
        'bm_xs', 'ep_xs', 'cfp_xs', 'sp_xs',
        # Quality
        'gpa_xs', 'roe_xs', 'roa_xs',
        # Risk
        'vol_12m_xs', 'ivol_xs', 'beta_xs',
        # Size
        'log_me_xs',
        # Analyst
        'sue_xs', 'revision_xs', 'beat_xs',
        # Turnover / Liquidity
        'turnover_xs', 'illiq_12m_xs',
    ]
    for c in core:
        if c in df_slice.columns:
            feat[c] = df_slice[c]

    # Pre-built interaction features
    interactions = [
        'mom_x_size_xs', 'val_x_prof_xs', 'mom_x_vol_xs',
        'ret_vs_sector_xs', 'bm_vs_sector_xs', 'ret_vs_ind_xs',
        'bm_vs_size_xs',
    ]
    for c in interactions:
        if c in df_slice.columns:
            feat[c] = df_slice[c]

    # Hand-crafted composites
    feat['quality_composite'] = (
        df_slice.get('gpa_xs', 0)
        + df_slice.get('roe_xs', 0)
        - df_slice.get('ag_xs', 0)
    ) / 3
    feat['earnings_momentum'] = (
        df_slice.get('sue_xs', 0)
        + df_slice.get('revision_xs', 0)
    ) / 2
    feat['reversal_mom_combo'] = (
        df_slice.get('ret_2_12_xs', 0)
        - df_slice.get('ret_1_xs', 0)
    )

    feat = feat.fillna(0.0)
    return feat


def build_features_ensemble(df_slice):
    """Tier 2: Broad feature set for RF and HGB (~60-70 features).
    Everything from Tier 1 + all remaining _xs features + extra composites.
    """
    # Start with all _xs features
    xs_cols = [c for c in df_slice.columns if c.endswith('_xs') and c != 'y_xs']
    feat = df_slice[xs_cols].copy()

    # Additional composites
    feat['quality_composite'] = (
        df_slice.get('gpa_xs', 0)
        + df_slice.get('roe_xs', 0)
        - df_slice.get('ag_xs', 0)
    ) / 3
    feat['earnings_momentum'] = (
        df_slice.get('sue_xs', 0)
        + df_slice.get('revision_xs', 0)
    ) / 2
    feat['reversal_mom_combo'] = (
        df_slice.get('ret_2_12_xs', 0)
        - df_slice.get('ret_1_xs', 0)
    )
    feat['peer_adj_earnings'] = (
        df_slice.get('sue_xs', 0)
        - df_slice.get('peer_sue_xs', 0)
    )
    feat['value_momentum'] = (
        df_slice.get('bm_xs', 0)
        + df_slice.get('ret_2_12_xs', 0)
    )

    feat = feat.fillna(0.0)
    return feat


print(f"Tier 1 features: {build_features_linear(df_sp).shape[1]}")
print(f"Tier 2 features: {build_features_ensemble(df_sp).shape[1]}")

In [ ]:
# ── Model Definitions ────────────────────────────────────────────────

from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

MODELS = {
    'Lasso': {
        'feature_builder': build_features_linear,
        'estimator': LassoCV(cv=5, max_iter=5000),
    },
    'RandomForest': {
        'feature_builder': build_features_ensemble,
        'estimator': RandomForestRegressor(
            n_estimators=300,
            max_depth=5,
            max_features=0.33,
            min_samples_leaf=50,
            n_jobs=-1,
            random_state=42,
        ),
    },
    'HGB': {
        'feature_builder': build_features_ensemble,
        'estimator': HistGradientBoostingRegressor(
            max_iter=300,
            max_depth=4,
            learning_rate=0.05,
            min_samples_leaf=100,
            early_stopping=False,
            random_state=42,
        ),
    },
}

print(f"Models defined: {list(MODELS.keys())}")

In [ ]:
# ── Walk-Forward Engine ──────────────────────────────────────────────
from sklearn.base import clone

def run_model(df_in, feature_builder, estimator, name, K=30):
    """Walk-forward backtest for a single (feature_builder, estimator) pair.

    Returns dict with:
      - 'long_only': pd.Series of monthly long-only returns
      - 'long_short': pd.Series of monthly long-short returns
    """
    all_months = sorted(df_in['ym'].unique())
    oos_months = [m for m in all_months if m >= OOS_START]

    long_only_rets = {}
    long_short_rets = {}

    # Determine retraining schedule
    retrain_months = oos_months[::RETRAIN_EVERY]  # every 12 months

    model = None
    for i, m in enumerate(oos_months):
        # Retrain if needed
        if m in retrain_months:
            train = df_in[df_in['ym'] < m].dropna(subset=[TRAIN_TARGET])
            X_train = feature_builder(train)
            y_train = train[TRAIN_TARGET]
            model = clone(estimator)
            model.fit(X_train, y_train)
            print(f"  [{name}] Trained on {len(train):,} rows up to {m}")

        # Predict this month
        test = df_in[df_in['ym'] == m].copy()
        if len(test) < 2 * K:
            continue

        X_test = feature_builder(test)
        test['pred'] = model.predict(X_test)

        # Long-only: top K stocks, equal weight
        top = test.nlargest(K, 'pred')
        lo_ret = top[EVAL_TARGET].mean()
        long_only_rets[m] = lo_ret

        # Long-short: top K minus bottom K (diagnostic)
        bot = test.nsmallest(K, 'pred')
        ls_ret = top[EVAL_TARGET].mean() - bot[EVAL_TARGET].mean()
        long_short_rets[m] = ls_ret

    return {
        'long_only': pd.Series(long_only_rets).sort_index(),
        'long_short': pd.Series(long_short_rets).sort_index(),
    }

print("Walk-forward engine loaded.")

In [ ]:
# ── Run All Models ───────────────────────────────────────────────────

results = {}

for name, cfg in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Running {name}...")
    print(f"{'='*50}")
    res = run_model(df_sp, cfg['feature_builder'], cfg['estimator'], name, K=K)
    results[name] = res
    # Quick preview
    lo = res['long_only']
    sr = lo.mean() / lo.std() * np.sqrt(12) if lo.std() > 0 else 0
    print(f"  → Long-only SR = {sr:.2f} ({len(lo)} months)")

print("\nAll models complete.")

In [ ]:
# ── Evaluation & Comparison ──────────────────────────────────────────

# --- Long-only performance (what matters for the grade) ---
print("=" * 60)
print("LONG-ONLY PORTFOLIO PERFORMANCE")
print("=" * 60)

lo_strats = {}
perf_rows = []

for name, res in results.items():
    lo = res['long_only']
    lo_strats[name] = lo
    perf_rows.append(perf(lo, name))

# Add SPY benchmark
perf_rows.append(perf(spy_oos, 'SPY'))

df_perf = pd.DataFrame(perf_rows).set_index('Strategy')
print(df_perf.to_string())

# Cumulative wealth plot - long only
plot_strats(lo_strats, 'Long-Only Portfolio: Cumulative Wealth vs SPY')

# --- Long-short diagnostics ---
print("\n" + "=" * 60)
print("LONG-SHORT DIAGNOSTICS (not for submission)")
print("=" * 60)

ls_perf_rows = []
for name, res in results.items():
    ls = res['long_short']
    ls_perf_rows.append(perf(ls, f'{name} L/S'))

df_ls_perf = pd.DataFrame(ls_perf_rows).set_index('Strategy')
print(df_ls_perf.to_string())

In [ ]:
# ── Optional Blend ───────────────────────────────────────────────────

def run_blend(df_in, models_cfg, K=30):
    """Walk-forward with averaged predictions from multiple models."""
    all_months = sorted(df_in['ym'].unique())
    oos_months = [m for m in all_months if m >= OOS_START]
    retrain_months = oos_months[::RETRAIN_EVERY]

    fitted_models = {}  # name -> fitted model
    long_only_rets = {}

    for i, m in enumerate(oos_months):
        # Retrain all models if needed
        if m in retrain_months:
            train = df_in[df_in['ym'] < m].dropna(subset=[TRAIN_TARGET])
            for name, cfg in models_cfg.items():
                X_train = cfg['feature_builder'](train)
                y_train = train[TRAIN_TARGET]
                mdl = clone(cfg['estimator'])
                mdl.fit(X_train, y_train)
                fitted_models[name] = (mdl, cfg['feature_builder'])
            print(f"  [Blend] Trained all models up to {m}")

        # Predict this month with each model, then average
        test = df_in[df_in['ym'] == m].copy()
        if len(test) < 2 * K:
            continue

        preds = []
        for name, (mdl, fb) in fitted_models.items():
            X_test = fb(test)
            preds.append(mdl.predict(X_test))

        test['pred_blend'] = np.mean(preds, axis=0)

        # Long-only
        top = test.nlargest(K, 'pred_blend')
        long_only_rets[m] = top[EVAL_TARGET].mean()

    return pd.Series(long_only_rets).sort_index()


# Find top 2 models by SR
model_srs = {}
for name, res in results.items():
    lo = res['long_only']
    model_srs[name] = lo.mean() / lo.std() * np.sqrt(12) if lo.std() > 0 else 0

sorted_models = sorted(model_srs.items(), key=lambda x: x[1], reverse=True)
top2_names = [n for n, _ in sorted_models[:2]]
print(f"Blending top 2 models: {top2_names}")

blend_cfg = {n: MODELS[n] for n in top2_names}
blend_rets = run_blend(df_sp, blend_cfg, K=K)

blend_sr = blend_rets.mean() / blend_rets.std() * np.sqrt(12)
print(f"Blend SR = {blend_sr:.2f}")
print(f"Best individual SR = {sorted_models[0][1]:.2f} ({sorted_models[0][0]})")
if blend_sr > sorted_models[0][1]:
    print("→ Blend IMPROVES over best individual model. Using blend.")
    results['Blend'] = {'long_only': blend_rets}
else:
    print("→ Blend does NOT improve. Sticking with best individual model.")

In [ ]:
# Re-plot with blend included (only if blend was added to results)
if 'Blend' in results:
    lo_strats_with_blend = {n: r['long_only'] for n, r in results.items() if 'long_only' in r}
    plot_strats(lo_strats_with_blend, 'All Strategies Including Blend vs SPY')
    print(perf(results['Blend']['long_only'], 'Blend'))

In [ ]:
# ── Lasso Feature Analysis ───────────────────────────────────────────

# Re-train Lasso on full pre-OOS data to inspect coefficients
train_full = df_sp[df_sp['ym'] < OOS_START].dropna(subset=[TRAIN_TARGET])
X_full = build_features_linear(train_full)
y_full = train_full[TRAIN_TARGET]

lasso_inspect = LassoCV(cv=5, max_iter=5000)
lasso_inspect.fit(X_full, y_full)

coefs = pd.Series(lasso_inspect.coef_, index=X_full.columns)
nonzero = coefs[coefs != 0].sort_values(key=abs, ascending=False)
print(f"Lasso selected {len(nonzero)} / {len(coefs)} features")
print(f"Best alpha: {lasso_inspect.alpha_:.6f}\n")
print("Non-zero coefficients (sorted by magnitude):")
print(nonzero.to_string())

In [ ]:
# ── Final Results ────────────────────────────────────────────────────

# Pick the best strategy (highest SR with MDD < 40%)
best_name = None
best_sr = -999

for name, res in results.items():
    if 'long_only' not in res:
        continue
    lo = res['long_only']
    sr = lo.mean() / lo.std() * np.sqrt(12) if lo.std() > 0 else 0
    cum = (1 + lo).cumprod()
    mdd = (cum / cum.cummax() - 1).min()
    if mdd > -0.40 and sr > best_sr:
        best_sr = sr
        best_name = name

print(f"BEST STRATEGY: {best_name} (SR = {best_sr:.2f})")
print(f"SPY Benchmark SR = {spy_sr:.2f}")
print(f"Alpha over SPY = {best_sr - spy_sr:+.2f} SR\n")

# Final performance table
best_lo = results[best_name]['long_only']
final_perf = pd.DataFrame([
    perf(best_lo, best_name),
    perf(spy_oos, 'SPY'),
]).set_index('Strategy')
print(final_perf.to_string())

# Final plot
plot_strats({best_name: best_lo}, f'Best Strategy ({best_name}) vs SPY')

### Task 2: Description of Approach

**Features:** We constructed two feature tiers from the cross-sectionally standardized (`_xs`) variables. Tier 1 (~29 features) for our linear model included core momentum (`ret_1_xs`, `ret_2_12_xs`), value (`bm_xs`, `ep_xs`), quality (`gpa_xs`, `roe_xs`), risk (`vol_12m_xs`, `ivol_xs`), and analyst signals (`sue_xs`, `revision_xs`), plus hand-crafted composites: a quality score (profitability minus asset growth), earnings momentum (SUE + revisions), and a reversal-momentum combination. Tier 2 (~97 features) for ensemble models added all remaining `_xs` features and additional composites (peer-adjusted earnings, value-momentum interaction).

**Models:** We tested three models: (1) LassoCV for automatic feature selection with L1 regularization, (2) Random Forest (300 trees, max_depth=5, min_samples_leaf=50) for non-linear patterns, and (3) HistGradientBoosting (300 iterations, max_depth=4, learning_rate=0.05) as our strongest learner. All models were deliberately configured with shallow depth and large leaf sizes to prevent overfitting on noisy return data (~3% signal).

**What worked:** [Fill in after running: best model name and SR, which features Lasso selected, whether blend helped, any surprising findings]

**What didn't work:** [Fill in after running: worst model and why, any features that didn't help, overfitting observations]

---
## Task 1: The Alpha Competition (60/100 points)

**Goal:** Build the best walk-forward alpha model. You are free to:
- Engineer new features (interactions, transforms, rolling stats, sector-relative, etc.)
- Choose any models, even those which are not taught.


**Rules:**
1. OOS from 2005-01 onward (no look-ahead)
2. Walk-forward only
3. Equal weight **LONG ONLY** portfolio.
3. Use `alpha_dataset_v2.csv` as base data (no external data)
4. Report  Sharpe and Maximum drawdown, annual return and compare with benchmarks.
5. Plot cumulative wealth plot with benchmark.   

**Scoring:** 20 of 60 points are ranked by OOS Sharpe vs all submissions.

**Perforamnce will be evaluated by SR (risk-free rate=0) given that MDD is under 40%:**
for example, student A SR=2 MDD=35%, student B SR=1.8, MDD=20%, then student A is better.

*Hint: Feature engineering contributes more to alpha than model complexity.*

### Task 2: Description of Approach (40/100 points)

*Describe your approach in 1 paragraph: what features did you engineer, what model
did you use, what hyperparameters, and why. What worked and what didn't.*

---
**Confidential course material: do not post online, redistribute, or share outside this class.**